# Modelo híbrido ML + ABM + SIG — Participación electoral, Localidad Riomar (Barranquilla)

Cuaderno de resultados del trabajo de grado. Todas las cifras, tablas y figuras de la sección de Resultados
deben salir de este cuaderno.

**Diseño:** entrenamiento y calibración con las elecciones de Alcaldía 2019; evaluación fuera de muestra con 2023.
Los modelos se estiman con toda la Localidad Riomar y se reportan aparte los puestos de la zona focal (5 barrios).

**Fuentes (todas oficiales):**
* Registraduría: resultados mesa a mesa (MMV) 2019 y 2023; DIVIPOLE 2019 y 2023 (potencial electoral, edad y sexo
  de los inscritos por puesto); DIVIPOLE 2023 georreferenciada (coordenadas de puestos).
* DANE: microdatos del CNPV 2018 (estrato, edad, sexo y nivel educativo por manzana) y cartografía MGN 2018.

Los archivos de entrada se generan con `prep_datos_riomar.py` y se ubican en la estructura de carpetas del proyecto.

## 0. Configuración

In [ ]:
BASE = '/content/drive/MyDrive/TesisRiomar'
ARCH_MANZANAS = '01_Cartografia_GIS/manzanas_estudio.gpkg'
ARCH_PUESTOS = '01_Cartografia_GIS/puestos.csv'                       # anio, puesto_id, nombre, lat, lon
ARCH_VARIABLES = '02_Datos_Sociodemograficos_DANE/manzanas_variables.csv'
ARCH_PUESTO_FEAT = '02_Datos_Sociodemograficos_DANE/puestos_features.csv'  # anio, puesto_id, variables DIVIPOLE
ARCH_MESAS = '03_Datos_Electorales_Registraduria/mesas.csv'           # anio, puesto_id, mesa, potencial, sufragantes
SALIDA = '04_Resultados'

ANIO_TRAIN, ANIO_TEST = 2019, 2023

# Zona focal (5 barrios): puestos que les sirven. Confirmar con conocimiento del territorio.
PUESTOS_FOCALES = {'R14': 'La Playa', 'R08': 'Las Flores', 'R07': 'San Salvador', 'R06': 'Siape', 'R09': 'Siape',
                   'R02': 'Villa Santos', 'R04': 'Villa Santos', 'R05': 'Villa Santos'}

# Variables por manzana (si se incorpora el CNPV 2018, agregar aquí sus columnas, en proporciones 0–1)
FEATURES_MANZANA = ['p_estrato_bajo', 'p_estrato_alto', 'p_edu_superior', 'p_20_29', 'p_65_mas', 'p_mujeres_cnpv']
# Variables por puesto (DIVIPOLE: composición de los inscritos)
FEATURES_PUESTO = ['p_mujeres', 'p_18_25', 'p_26_40', 'p_41_60', 'p_60_mas']

CRS_METRICO = 'EPSG:9377'
SEMILLA = 42
N_REPLICAS = 100
T_TICKS = 50
ALPHA = 0.8
RADIO_M = 300
RADIOS_SENS = [100, 200, 300, 400, 500]
ALPHAS_SENS = [0.7, 0.8, 0.9]
N_REPLICAS_SENS = 30
DELTA = 1e-4
TASA_MIN, TASA_MAX = 0.15, 1.0   # mesas fuera de este rango se excluyen como atípicas
N_BOOT = 2000

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy import sparse
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import statsmodels.api as sm
import geopandas as gpd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive  # noqa
    drive.mount('/content/drive')
except Exception:
    pass
P = lambda f: os.path.join(BASE, f)
os.makedirs(P(SALIDA), exist_ok=True)
OUT = lambda f: os.path.join(BASE, SALIDA, f)
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})

## 1. Carga de datos

In [ ]:
mz = gpd.read_file(P(ARCH_MANZANAS)); mz['manzana_id'] = mz.manzana_id.astype(str)
var = pd.read_csv(P(ARCH_VARIABLES), dtype={'manzana_id': str})
puestos = pd.read_csv(P(ARCH_PUESTOS), dtype={'puesto_id': str})
pfeat = pd.read_csv(P(ARCH_PUESTO_FEAT), dtype={'puesto_id': str})
mesas = pd.read_csv(P(ARCH_MESAS), dtype={'puesto_id': str})
var = var.drop(columns=[c for c in var.columns if c in mz.columns and c != 'manzana_id'])
mz = mz.merge(var, on='manzana_id', how='inner')
mesas['tasa'] = mesas.sufragantes / mesas.potencial
atip = mesas[(mesas.tasa < TASA_MIN) | (mesas.tasa > TASA_MAX)]
print(f'Mesas atípicas excluidas: {len(atip)}'); print(atip[['anio', 'puesto_id', 'mesa', 'potencial', 'sufragantes']].to_string(index=False))
mesas = mesas.drop(atip.index).reset_index(drop=True)
print(f'Manzanas: {len(mz)} | Agentes (población de 18 años o más, CNPV 2018): {int(mz.pob_18.sum()):,}')
print(mesas.groupby('anio').agg(puestos=('puesto_id', 'nunique'), mesas=('mesa', 'size'), potencial=('potencial', 'sum'), sufragantes=('sufragantes', 'sum')))

## 2. Áreas de influencia por año (Voronoi) y variables por puesto — OE1

In [ ]:
mzm = mz.to_crs(CRS_METRICO)
cent = np.c_[mzm.geometry.centroid.x, mzm.geometry.centroid.y]
mzm['densidad'] = mzm.pob_18 / (mzm.geometry.area / 1e4).clip(lower=0.01)
FEAT_MZ = FEATURES_MANZANA + ['dist_puesto_m', 'densidad']

def areas_influencia(anio):
    pa = puestos[puestos.anio == anio]
    pg = gpd.GeoDataFrame(pa, geometry=gpd.points_from_xy(pa.lon, pa.lat), crs='EPSG:4326').to_crs(CRS_METRICO)
    dist, idx = cKDTree(np.c_[pg.geometry.x, pg.geometry.y]).query(cent)
    return pa.puesto_id.values[idx], dist, pg

CATCH = {}
feat_p = []
for y in (ANIO_TRAIN, ANIO_TEST):
    pid, dist, pg = areas_influencia(y)
    CATCH[y] = dict(puesto=pid, dist=dist, pg=pg)
    tmp = mzm[['manzana_id', 'pob_18', 'densidad'] + FEATURES_MANZANA].copy(); tmp['puesto_id'] = pid; tmp['dist_puesto_m'] = dist
    g = tmp.groupby('puesto_id').apply(lambda g: pd.Series({c: np.average(g[c], weights=g.pob_18.clip(lower=1)) for c in FEAT_MZ}
                                                           | {'pob_area': g.pob_18.sum(), 'n_manzanas': len(g)})).reset_index()
    g['anio'] = y; feat_p.append(g)
feat_p = pd.concat(feat_p).merge(pfeat, on=['anio', 'puesto_id'], how='left')

mesas['mesa_rank'] = mesas.groupby(['anio', 'puesto_id']).mesa.rank(pct=True)
mesas['log_potencial'] = np.log(mesas.potencial)
D = mesas.merge(feat_p, on=['anio', 'puesto_id'], how='left')
X_COLS = FEAT_MZ + FEATURES_PUESTO + ['mesa_rank', 'log_potencial']
assert D[X_COLS].isna().sum().sum() == 0, D[X_COLS].isna().sum()

def reemplazo_ceros(y):
    y = np.asarray(y, float)
    return np.where(y <= 0, DELTA, np.where(y >= 1, 1 - DELTA, y))
ilr = lambda y: np.log(reemplazo_ceros(y) / (1 - reemplazo_ceros(y))) / np.sqrt(2)
ilr_inv = lambda z: 1 / (1 + np.exp(-np.sqrt(2) * z))
D['z'] = ilr(D.tasa)
tr = D[D.anio == ANIO_TRAIN].reset_index(drop=True)
te = D[D.anio == ANIO_TEST].reset_index(drop=True)
print(f'Mesas entrenamiento {ANIO_TRAIN}: {len(tr)} ({tr.puesto_id.nunique()} puestos) | evaluación {ANIO_TEST}: {len(te)} ({te.puesto_id.nunique()} puestos)')

### 2.1 OE1: participación por puesto, correlaciones y autocorrelación espacial

In [ ]:
pp_obs = mesas.groupby(['anio', 'puesto_id']).agg(potencial=('potencial', 'sum'), sufragantes=('sufragantes', 'sum'), mesas=('mesa', 'size')).reset_index()
pp_obs['tasa'] = pp_obs.sufragantes / pp_obs.potencial
desc = pp_obs.pivot(index='puesto_id', columns='anio', values='tasa').add_prefix('tasa_').reset_index()
desc = desc.merge(puestos[puestos.anio == ANIO_TEST][['puesto_id', 'nombre']], on='puesto_id', how='left')
desc['zona_focal'] = desc.puesto_id.map(PUESTOS_FOCALES).fillna('')
print(desc.round(3).to_string(index=False))
resumen_part = pp_obs.groupby('anio').apply(lambda g: pd.Series({'participacion': g.sufragantes.sum() / g.potencial.sum(),
                                                                  'min_puesto': g.tasa.min(), 'max_puesto': g.tasa.max()})).round(4)
print(resumen_part)

ftr = feat_p[feat_p.anio == ANIO_TRAIN].merge(pp_obs[pp_obs.anio == ANIO_TRAIN][['puesto_id', 'tasa']], on='puesto_id')
corr = ftr[['tasa'] + FEAT_MZ + FEATURES_PUESTO].corr(method='spearman')['tasa'].drop('tasa').sort_values()
print(f'Spearman con la participación {ANIO_TRAIN} (n = {len(ftr)} puestos):'); print(corr.round(3))
corr_mesa = tr[['tasa'] + X_COLS].corr(method='spearman')['tasa'].drop('tasa').sort_values()

moran_res = {}
try:
    from libpysal.weights import KNN
    from esda.moran import Moran, Moran_Local
    for y in (ANIO_TRAIN, ANIO_TEST):
        pg = CATCH[y]['pg'].merge(pp_obs[pp_obs.anio == y][['puesto_id', 'tasa']], on='puesto_id')
        w = KNN.from_dataframe(pg, k=3); w.transform = 'r'
        mi = Moran(pg.tasa.values, w, permutations=999)
        moran_res[y] = dict(I=round(float(mi.I), 3), p_sim=round(float(mi.p_sim), 3), n=len(pg))
        if y == ANIO_TEST:
            ml = Moran_Local(pg.tasa.values, w, permutations=999, seed=SEMILLA)
            lab = {1: 'Alto-Alto', 2: 'Bajo-Alto', 3: 'Bajo-Bajo', 4: 'Alto-Bajo'}
            pg['LISA'] = [lab[q] if p < 0.05 else 'No significativo' for q, p in zip(ml.q, ml.p_sim)]
            pg_lisa = pg
    print('I de Moran global por puesto (k = 3 vecinos):', moran_res)
except Exception as ex:
    print('Moran no disponible:', ex); pg_lisa = None

# Figura 1: área de estudio y áreas de influencia 2023
fig, ax = plt.subplots(figsize=(9, 7.5))
m23 = mzm.assign(puesto_id=CATCH[ANIO_TEST]['puesto']).to_crs('EPSG:4326')
m23.plot(ax=ax, column='puesto_id', categorical=True, cmap='tab20', edgecolor='white', linewidth=0.2, alpha=0.55)
pg = CATCH[ANIO_TEST]['pg'].to_crs('EPSG:4326')
foc = pg.puesto_id.isin(PUESTOS_FOCALES)
pg[~foc].plot(ax=ax, color='#1f4e78', markersize=35, edgecolor='white', zorder=3)
pg[foc].plot(ax=ax, color='#c0392b', markersize=60, marker='*', edgecolor='white', zorder=4)
for _, r in pg.iterrows():
    ax.annotate(r.puesto_id, (r.geometry.x, r.geometry.y), fontsize=7, xytext=(3, 3), textcoords='offset points')
ax.set_xlabel('Longitud'); ax.set_ylabel('Latitud')
ax.set_title('Localidad Riomar: manzanas por área de influencia (Voronoi) y puestos 2023\n(estrella roja: puestos de la zona focal)', fontsize=10)
plt.tight_layout(); plt.savefig(OUT('figura1_area_estudio.png'), dpi=300); plt.show()

# Mapa de participación 2023 por puesto
fig, ax = plt.subplots(figsize=(8, 6.5))
mzm.to_crs('EPSG:4326').plot(ax=ax, color='#eeeeee', edgecolor='white', linewidth=0.2)
t23 = pg.merge(pp_obs[pp_obs.anio == ANIO_TEST][['puesto_id', 'tasa']], on='puesto_id')
t23.plot(ax=ax, column='tasa', cmap='viridis', markersize=t23.tasa * 160, legend=True, edgecolor='black',
         legend_kwds={'label': f'Participación {ANIO_TEST}', 'shrink': 0.6})
ax.set_title(f'Participación electoral por puesto, Alcaldía {ANIO_TEST}', fontsize=10)
plt.tight_layout(); plt.savefig(OUT('figura_participacion_puestos.png'), dpi=300); plt.show()

## 3. Modelos de referencia (entrenados con 2019, evaluados en 2023)

In [ ]:
def modelos():
    return {
        'Regresión logística': None,
        'Random Forest': RandomForestRegressor(n_estimators=500, min_samples_leaf=5, random_state=SEMILLA, n_jobs=-1),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, max_depth=2, learning_rate=0.05, subsample=0.8, random_state=SEMILLA),
        'Red neuronal feedforward (FNN)': make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(16,), alpha=1e-2, max_iter=5000, random_state=SEMILLA)),
    }

def ajustar(nombre, mdl, dtr):
    if nombre == 'Regresión logística':
        sc = StandardScaler().fit(dtr[X_COLS])
        Xg = sm.add_constant(sc.transform(dtr[X_COLS]), has_constant='add')
        g = sm.GLM(reemplazo_ceros(dtr.tasa), Xg, family=sm.families.Binomial(), var_weights=dtr.potencial).fit_regularized(alpha=1e-3, L1_wt=0.0)
        return lambda X: g.predict(sm.add_constant(sc.transform(X[X_COLS]), has_constant='add'))
    m = mdl.fit(dtr[X_COLS], dtr.z)
    return lambda X: ilr_inv(m.predict(X[X_COLS]))

pred = pd.DataFrame({'puesto_id': te.puesto_id, 'mesa': te.mesa, 'potencial': te.potencial, 'observado': te.tasa})
p19 = tr.set_index(['puesto_id', 'mesa']).tasa
p19p = tr.groupby('puesto_id').apply(lambda g: g.sufragantes.sum() / g.potencial.sum())
p19loc = tr.sufragantes.sum() / tr.potencial.sum()
NOM_P = 'Persistencia (tasa de 2019)'
pred[NOM_P] = [p19.get((a, b), p19p.get(a, p19loc)) for a, b in zip(te.puesto_id, te.mesa)]

predictores = {}
for nom, mdl in modelos().items():
    predictores[nom] = ajustar(nom, mdl, tr)
    pred[nom] = predictores[nom](te)

cv = {}
for nom in modelos():
    errs = []
    for pid in tr.puesto_id.unique():
        a, b = tr[tr.puesto_id != pid], tr[tr.puesto_id == pid]
        errs.append(np.abs(ajustar(nom, modelos()[nom], a)(b) - b.tasa.values))
    cv[nom] = float(np.mean(np.concatenate(errs)) * 100)
cv = pd.Series(cv, name='MAE LOPO-CV 2019 (p. p.)').sort_values()
BASE_ML = cv.index[0]
print(cv.round(2)); print('Clasificador base del ABM:', BASE_ML)

## 4. Modelo basado en agentes (OE2)

In [ ]:
rng0 = np.random.default_rng(SEMILLA)
xs, ys, mids = [], [], []
for _, r in mzm.iterrows():
    n = int(r.pob_18)
    if n <= 0: continue
    minx, miny, maxx, maxy = r.geometry.bounds; got = 0
    while got < n:
        k = 2 * (n - got) + 10
        cx = rng0.uniform(minx, maxx, k); cy = rng0.uniform(miny, maxy, k)
        ok = gpd.GeoSeries(gpd.points_from_xy(cx, cy)).within(r.geometry).values
        cx, cy = cx[ok][:n - got], cy[ok][:n - got]
        xs.append(cx); ys.append(cy); mids += [r.manzana_id] * len(cx); got += len(cx)
ag = pd.DataFrame({'manzana_id': mids, 'x': np.concatenate(xs), 'y': np.concatenate(ys)})
mz_idx = pd.Series(np.arange(len(mzm)), index=mzm.manzana_id.values)
ag_mz = mz_idx.reindex(ag.manzana_id).values
ag['puesto_id'] = CATCH[ANIO_TEST]['puesto'][ag_mz]
ag['dist_puesto_m'] = CATCH[ANIO_TEST]['dist'][ag_mz]
for c in ['densidad'] + FEATURES_MANZANA: ag[c] = mzm[c].values[ag_mz]
n_mz = np.bincount(ag_mz, minlength=len(mzm)).astype(float)
print(f'Agentes: {len(ag):,} en {int((n_mz > 0).sum())} manzanas')

def p_ml_agentes(f, d_anio):
    X = ag[FEAT_MZ].copy()
    pf = pfeat[pfeat.anio == d_anio.anio.iloc[0]].set_index('puesto_id')
    for c in FEATURES_PUESTO: X[c] = ag.puesto_id.map(pf[c]).values
    X['mesa_rank'] = 0.5
    X['log_potencial'] = ag.puesto_id.map(d_anio.groupby('puesto_id').log_potencial.mean()).fillna(d_anio.log_potencial.mean()).values
    return np.clip(f(X), 1e-4, 1 - 1e-4)

def vecinos(radio):
    dm = cKDTree(cent).sparse_distance_matrix(cKDTree(cent), radio, output_type='coo_matrix')
    A = sparse.csr_matrix((np.ones(dm.nnz), (dm.row, dm.col)), shape=(len(cent), len(cent))) + sparse.identity(len(cent), format='csr')
    A.data[:] = 1.0
    return A

def simular(p_ml, A, alpha, n_rep, ticks=T_TICKS, guardar_curva=False):
    upid = np.unique(ag.puesto_id.values); inv = np.searchsorted(upid, ag.puesto_id.values)
    tasas = np.zeros((n_rep, len(upid))); curvas = np.zeros((n_rep, ticks + 1)) if guardar_curva else None
    den = A @ n_mz
    for r in range(n_rep):
        rng = np.random.default_rng(SEMILLA + r)
        P = p_ml.copy()
        if guardar_curva: curvas[r, 0] = P.mean()
        for t in range(ticks):
            b = (rng.random(len(P)) < P).astype(float)
            pv_mz = (A @ np.bincount(ag_mz, weights=b, minlength=len(mzm))) / np.maximum(den, 1)
            P = alpha * p_ml + (1 - alpha) * pv_mz[ag_mz]
            if guardar_curva: curvas[r, t + 1] = P.mean()
        voto = rng.random(len(P)) < P
        tasas[r] = np.bincount(inv, weights=voto, minlength=len(upid)) / np.bincount(inv, minlength=len(upid))
    return pd.DataFrame(tasas, columns=upid), curvas

f_base = predictores[BASE_ML]
pml_te = p_ml_agentes(f_base, te)
A300 = vecinos(RADIO_M)
print('Agentes vecinos promedio dentro de %d m: %.0f' % (RADIO_M, np.average(A300 @ n_mz, weights=np.maximum(n_mz, 1e-9))))
sim_h, curvas = simular(pml_te, A300, ALPHA, N_REPLICAS, guardar_curva=True)
sim_1, _ = simular(pml_te, A300, 1.0, N_REPLICAS)

lg = lambda p: np.log(p / (1 - p))
base_p = pd.Series(pml_te).groupby(ag.puesto_id.values).mean()
def a_mesas(sim, nombre):
    abm = sim.median().clip(1e-4, 1 - 1e-4)
    ajuste = (lg(abm) - lg(base_p)).reindex(te.puesto_id).fillna(0).values
    pred[nombre] = 1 / (1 + np.exp(-(lg(np.clip(pred[BASE_ML].values, 1e-4, 1 - 1e-4)) + ajuste)))

NOM_A1 = 'Clasificador base sin interacción vecinal (α = 1)'
NOM_H = f'Modelo híbrido ML + ABM + SIG (r = {RADIO_M} m, α = {ALPHA})'
a_mesas(sim_1, NOM_A1); a_mesas(sim_h, NOM_H)

fig, ax = plt.subplots(figsize=(8, 7))
mzm.plot(ax=ax, color='#f2f2f2', edgecolor='#c8c8c8', linewidth=0.2)
s = ag.sample(min(len(ag), 30000), random_state=SEMILLA)
ax.scatter(s.x, s.y, s=0.2, c='teal', alpha=0.4)
CATCH[ANIO_TEST]['pg'].plot(ax=ax, color='black', marker='^', markersize=30)
ax.set_axis_off(); ax.set_title(f'Población sintética: {len(ag):,} agentes en {int((n_mz > 0).sum())} manzanas (muestra de {len(s):,})', fontsize=10)
plt.tight_layout(); plt.savefig(OUT('figura3_agentes.png'), dpi=300); plt.show()

fig, ax = plt.subplots(figsize=(8, 4.2))
t = np.arange(0, T_TICKS + 1)
ax.plot(t, np.median(curvas, 0), color='#1f4e78', label='Mediana de las réplicas')
ax.fill_between(t, np.percentile(curvas, 2.5, 0), np.percentile(curvas, 97.5, 0), color='#1f4e78', alpha=0.25, label='Intervalo empírico 95 %')
ax.set_xlabel('Tick'); ax.set_ylabel('Probabilidad media de participación'); ax.legend()
plt.tight_layout(); plt.savefig(OUT('figura4_convergencia.png'), dpi=300); plt.show()

## 5. Tabla 9 — Evaluación fuera de muestra 2023 (OE3)

In [ ]:
umbral = tr.tasa.median()
MODELOS = [NOM_P, 'Regresión logística', 'Random Forest', 'Gradient Boosting', 'Red neuronal feedforward (FNN)', NOM_A1, NOM_H]
def metricas(obs, pr):
    e = (pr - obs) * 100; yo, yp = obs > umbral, pr > umbral
    return {'MAE (p. p.)': np.mean(np.abs(e)), 'RMSE (p. p.)': np.sqrt(np.mean(e ** 2)), 'Sesgo (p. p.)': np.mean(e),
            'Accuracy': accuracy_score(yo, yp), 'Precision': precision_score(yo, yp, zero_division=0),
            'Recall': recall_score(yo, yp, zero_division=0), 'F1': f1_score(yo, yp, zero_division=0)}
tabla9 = pd.DataFrame({m: metricas(pred.observado.values, pred[m].values) for m in MODELOS}).T
tabla9.index.name = 'Modelo'
print(f'n = {len(pred)} mesas, {pred.puesto_id.nunique()} puestos; umbral alta/baja = mediana {ANIO_TRAIN} = {umbral:.3f}')
print(tabla9.round(3).to_string())

foc = pred.puesto_id.isin(PUESTOS_FOCALES).values
tabla9_focal = pd.DataFrame({m: metricas(pred.observado.values[foc], pred[m].values[foc]) for m in MODELOS}).T[['MAE (p. p.)', 'RMSE (p. p.)', 'Sesgo (p. p.)']]
print(f'Zona focal: {foc.sum()} mesas en {pred[foc].puesto_id.nunique()} puestos'); print(tabla9_focal.round(3).to_string())

# Puestos nuevos en 2023 (sin historia 2019) frente a puestos con historia: prueba de generalización
nuevo = ~pred.puesto_id.isin(tr.puesto_id.unique()).values
tabla9_nuevos = pd.concat({
    'Con historia 2019': pd.DataFrame({m: metricas(pred.observado.values[~nuevo], pred[m].values[~nuevo]) for m in MODELOS}).T[['MAE (p. p.)', 'RMSE (p. p.)', 'Sesgo (p. p.)']],
    'Nuevos en 2023': pd.DataFrame({m: metricas(pred.observado.values[nuevo], pred[m].values[nuevo]) for m in MODELOS}).T[['MAE (p. p.)', 'RMSE (p. p.)', 'Sesgo (p. p.)']]})
print(f'Puestos nuevos: {pred[nuevo].puesto_id.nunique()} ({nuevo.sum()} mesas)'); print(tabla9_nuevos.round(3).to_string())

# Métricas de la zona focal por barrio
foc_barrio = pd.concat({b: pd.DataFrame({m: metricas(pred.observado.values[pred.puesto_id.map(PUESTOS_FOCALES).eq(b).values],
                                                      pred[m].values[pred.puesto_id.map(PUESTOS_FOCALES).eq(b).values]) for m in [mejor_ref if 'mejor_ref' in dir() else 'Random Forest', NOM_H]}).T[['MAE (p. p.)', 'Sesgo (p. p.)']]
                        for b in sorted(set(PUESTOS_FOCALES.values()))})
print(foc_barrio.round(3).to_string())

pp = pred.groupby('puesto_id').apply(lambda g: pd.Series({c: np.average(g[c], weights=g.potencial) for c in ['observado'] + MODELOS}))
tabla9_puesto = pd.DataFrame({m: metricas(pp.observado.values, pp[m].values) for m in MODELOS}).T[['MAE (p. p.)', 'RMSE (p. p.)', 'Sesgo (p. p.)']]

refs = [m for m in MODELOS if m != NOM_H]
mejor_ref = tabla9.loc[refs, 'MAE (p. p.)'].idxmin()
rngb = np.random.default_rng(SEMILLA)
pids = pred.puesto_id.unique(); grupos = {p: pred.index[pred.puesto_id == p].values for p in pids}
def dif_boot(a, b):
    out = []
    for _ in range(N_BOOT):
        ii = np.concatenate([grupos[p] for p in rngb.choice(pids, len(pids))])
        out.append(np.mean(np.abs(pred.loc[ii, a] - pred.loc[ii, 'observado'])) - np.mean(np.abs(pred.loc[ii, b] - pred.loc[ii, 'observado'])))
    return np.percentile(np.array(out) * 100, [2.5, 97.5])
d_obs = tabla9.loc[NOM_H, 'MAE (p. p.)'] - tabla9.loc[mejor_ref, 'MAE (p. p.)']; ic = dif_boot(NOM_H, mejor_ref)
d_a1 = tabla9.loc[NOM_H, 'MAE (p. p.)'] - tabla9.loc[NOM_A1, 'MAE (p. p.)']; ic_a1 = dif_boot(NOM_H, NOM_A1)
print(f'Híbrido − {mejor_ref}: {d_obs:.2f} p. p. (IC 95 % bootstrap por puestos [{ic[0]:.2f}, {ic[1]:.2f}])')
print(f'Híbrido − α=1: {d_a1:.3f} p. p. (IC 95 % [{ic_a1[0]:.3f}, {ic_a1[1]:.3f}])')

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(pred.observado * 100, pred[NOM_H] * 100, s=10, alpha=0.6, label='Híbrido')
ax.scatter(pred.observado * 100, pred[mejor_ref] * 100, s=10, alpha=0.5, marker='x', label=mejor_ref)
lim = [pred.observado.min() * 100 - 3, pred.observado.max() * 100 + 3]
ax.plot(lim, lim, 'k--', lw=1); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel(f'Participación observada {ANIO_TEST} (%)'); ax.set_ylabel('Participación estimada (%)'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT('figura_observado_vs_estimado.png'), dpi=300); plt.show()

## 6. Sensibilidad (radio de vecindad y α)

In [ ]:
sens = []
for rad in RADIOS_SENS:
    A = A300 if rad == RADIO_M else vecinos(rad)
    for a in ALPHAS_SENS:
        simr, _ = simular(pml_te, A, a, N_REPLICAS_SENS)
        nm = f'_r{rad}_a{a}'; a_mesas(simr, nm)
        mm = metricas(pred.observado.values, pred[nm].values)
        sens.append({'radio_m': rad, 'alpha': a, 'MAE (p. p.)': mm['MAE (p. p.)'], 'RMSE (p. p.)': mm['RMSE (p. p.)']})
        pred.drop(columns=nm, inplace=True)
sens = pd.DataFrame(sens)
fig, ax = plt.subplots(figsize=(7, 4.2))
for a, g in sens.groupby('alpha'): ax.plot(g.radio_m, g['MAE (p. p.)'], marker='o', label=f'α = {a}')
ax.axhline(tabla9.loc[NOM_A1, 'MAE (p. p.)'], color='gray', ls='--', label='Sin interacción (α = 1)')
ax.set_xlabel('Radio de vecindad (m)'); ax.set_ylabel(f'MAE {ANIO_TEST} (p. p.)'); ax.legend()
plt.tight_layout(); plt.savefig(OUT('figura5_sensibilidad.png'), dpi=300); plt.show()
print(sens.round(3).to_string(index=False))

## 7. Exportar

In [ ]:
resumen = {
    'n_manzanas': int(len(mzm)), 'n_agentes': int(len(ag)), 'mesas_atipicas_excluidas': int(len(atip)),
    'n_mesas_train': int(len(tr)), 'n_puestos_train': int(tr.puesto_id.nunique()),
    'n_mesas_test': int(len(te)), 'n_puestos_test': int(te.puesto_id.nunique()),
    'participacion': {int(k): round(float(v), 4) for k, v in resumen_part.participacion.items()},
    'umbral_alta_baja': round(float(umbral), 4), 'clasificador_base': BASE_ML, 'cv_2019': cv.round(3).to_dict(),
    'moran': moran_res, 'mejor_referencia': mejor_ref,
    'MAE_hibrido': round(float(tabla9.loc[NOM_H, 'MAE (p. p.)']), 3), 'RMSE_hibrido': round(float(tabla9.loc[NOM_H, 'RMSE (p. p.)']), 3),
    'MAE_mejor_ref': round(float(tabla9.loc[mejor_ref, 'MAE (p. p.)']), 3), 'RMSE_mejor_ref': round(float(tabla9.loc[mejor_ref, 'RMSE (p. p.)']), 3),
    'dif_MAE_hibrido_vs_mejor_ref': round(float(d_obs), 3), 'IC95_dif': [round(float(x), 3) for x in ic],
    'dif_MAE_hibrido_vs_alpha1': round(float(d_a1), 3), 'IC95_dif_alpha1': [round(float(x), 3) for x in ic_a1],
    'MAE_hibrido_puestos_nuevos': round(float(tabla9_nuevos.loc[('Nuevos en 2023', NOM_H), 'MAE (p. p.)']), 3),
    'MAE_persistencia_puestos_nuevos': round(float(tabla9_nuevos.loc[('Nuevos en 2023', NOM_P), 'MAE (p. p.)']), 3),
    'MAE_hibrido_zona_focal': round(float(tabla9_focal.loc[NOM_H, 'MAE (p. p.)']), 3),
    'parametros': dict(semilla=SEMILLA, replicas=N_REPLICAS, ticks=T_TICKS, alpha=ALPHA, radio_m=RADIO_M),
}
with open(OUT('resumen_resultados.json'), 'w', encoding='utf-8') as fh: json.dump(resumen, fh, ensure_ascii=False, indent=2)
with pd.ExcelWriter(OUT('resultados_tesis.xlsx')) as xw:
    tabla9.round(3).to_excel(xw, sheet_name='Tabla9_mesa')
    tabla9_focal.round(3).to_excel(xw, sheet_name='Tabla9_zona_focal')
    tabla9_puesto.round(3).to_excel(xw, sheet_name='Tabla9_puesto')
    tabla9_nuevos.round(3).to_excel(xw, sheet_name='Tabla9_nuevos_vs_historia')
    foc_barrio.round(3).to_excel(xw, sheet_name='Zona_focal_por_barrio')
    cv.round(3).to_frame().to_excel(xw, sheet_name='CV_2019')
    sens.round(3).to_excel(xw, sheet_name='Sensibilidad', index=False)
    desc.round(4).to_excel(xw, sheet_name='OE1_puestos', index=False)
    corr.round(3).to_frame('spearman_puesto').to_excel(xw, sheet_name='OE1_corr_puesto')
    corr_mesa.round(3).to_frame('spearman_mesa').to_excel(xw, sheet_name='OE1_corr_mesa')
    feat_p.round(4).to_excel(xw, sheet_name='Variables_puesto', index=False)
    atip.to_excel(xw, sheet_name='Mesas_atipicas', index=False)
    pred.round(4).to_excel(xw, sheet_name='Predicciones_2023', index=False)
print(json.dumps(resumen, ensure_ascii=False, indent=2))